# Week 10: Tuning clustering hyperparameters

Last week we learned a few unsupervised clustering algorithms. Each of these require some pre-defined settings, also called settings or hyperparameters. Today we will learn how to compare models between different hyperparameter choices. 

Brief aside: parameters vs hyperparameters? 
* **Parameters** are determined by the data through the modeling process. 
* **Hyperparamters** are set by you before any modeling occurs

So, in k-means, the k clusters we choose is a **hyperparameter**, but the locations of the cluster centroids are **parameters**. 

## 1 The elbow method

The first way to look for the right number of clusters is through the "elbow method". When we run KMeans, we are trying to group points so that points in the same cluster are as similar as possible.

To measure this, we compute the Sum of Squared Errors (SSE), also called inertia in Python. SSE measures how tightly packed the clusters are. Low SSE = points are close to their cluster centers (good) while a high SSE = points are spread out (bad). 

KMeans is designed to minimize SSE. We can try different values of k to see how much better the clustering gets as we increase k. However! SSE will always decrease as k increases (more clusters = smaller, tighter groups).

In [ ]:
#Imports
import pandas as pd 
import numpy as np 
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import robust_scale, scale, minmax_scale
from sklearn.neighbors import NearestNeighbors

In [ ]:
# First we will work with synthetic data

# Create synthetic dataset
X, y_true = sklearn.datasets.make_blobs(n_samples=300,
                                        centers=4, 
                                        cluster_std=1.2,
                                        random_state=42)

# Plot the data
plt.scatter(X[:, 0], X[:, 1])
plt.title("Synthetic Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
# A list holds the SSE values for each k
sse = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=0)
    kmeans.fit(X)
    sse.append(kmeans.inertia_) # These are the SSEs

In [ ]:
# We are looking for the elbow in the curve
# This is where adding more clusters doesn't do much more for the SSE

fig, ax = plt.subplots(figsize=(6,6))
plt.plot(range(1, 11), sse)
plt.xticks(range(1, 11))
plt.xlabel("Number of Clusters")
plt.ylabel("SSE")
plt.show()

In [ ]:
## YOUR TURN 
## Create new clusters/blobs with a different k 

## Our real data without any known k

In [ ]:
# Let's read in data from the EPA on environmental justice again

ej = pd.read_csv('EJI_2024_United_States.csv')
ej.head()

In [ ]:
# We'll keep only a few columns of interest right now
cols_of_interest = ["STATEABBR", # state abbreviation code
  "GEOID_2020", # census tract information - keep for plotting
  "E_POV200", # % below 200% poverty
  "E_NOHSDP", # % without HS diploma
  "E_UNEMP", # % unemployed
  "E_RENTER", # % renter-occupied homes
  "E_HOUBDN", # % housing cost burdened (make <$75k and pay >30% in housing costs)
  "E_MINRTY", # % minority population
  "E_LIMENG", # % limited English proficiency
  "E_PARK", # % of people within 1-mi of a green space
  "E_ROAD", # % of people within 1-mi of a high density road
  "E_DSLPM", # concentrations of diesel particulate matter
  "E_TOTCR"] # the percentile likelihood of developing cancer over ones lifetime 

ej_subset = ej.loc[:, cols_of_interest]
ej_subset.head()

In [ ]:
# Some of the "missing" data is input as -999
ej_subset = ej_subset.replace(-999, np.nan)

# Now we can remove rows that are missing key values
ej_subset = ej_subset.dropna()

# We also need the GEOID to be a string value
ej_subset['GEOID_2020'] = ej_subset['GEOID_2020'].astype(str)
ej_subset.describe()

In [ ]:
# Finally, let's focus on one state 
texas = ej_subset[ej_subset['STATEABBR'] == 'TX']

# And grab the geometry information from the census
# We need the geometries from the census 
year=2020
state_fips='48'
url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{state_fips}_tract.zip"
tracts = gpd.read_file(url)

texas = pd.merge(left=texas, right=tracts, how='left', left_on='GEOID_2020', right_on='GEOID')
texas = gpd.GeoDataFrame(texas,
                         geometry='geometry', 
                         crs='EPSG:4326')


# And separate the GEO information for later
variables = cols_of_interest[2:]
texas_data = texas[variables]
texas_data.head()

In [ ]:
# We can scale our texas subset
texas_scaled = robust_scale(texas_data)
texas_scaled

In [ ]:
## YOUR TURN
## Run the elblow method with our real data



## 2 Silhouette Scores

The silhouette score measures how well each point fits into its cluster, compared to other clusters.

For each point, we compute the average distance to the same cluster and the average distance to points in the nearest different cluster. 

The scores are between -1 (wrong cluster assignment) and 1 (very well clustered). 0 means there are overlapping clusters. 

In [ ]:
# Using our synthetic data

silhouette_scores = []

# silhouette score is undefined for k=1
k_values = range(2, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)

# Plot silhouette scores
plt.plot(k_values, silhouette_scores, marker='o')
plt.title("Silhouette Scores")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.show()

# 3 k-Distance Graph (for DBSCAN)

The k-distance graph is a tool used to help choose the epsilon parameter in DBSCAN by showing how far each point is from its k-th nearest neighbor (where k = min_samples). For every point, we compute this distance, sort all the values from smallest to largest, and plot them. In dense clusters, these distances are relatively small and change slowly, but as we move to points in sparser regions or noise, the distances increase sharply. We look for this sudden bend (or “knee”) in the plot and choose epsilon just before that increase, since it represents a natural cutoff between dense clusters and outliers.

In [ ]:
# Set min_samples (k)
k = 5

# Fit nearest neighbors model
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X)

# Get distances to k-th nearest neighbor
distances, indices = neighbors_fit.kneighbors(X)

# Take the k-th nearest distance (last column)
k_distances = distances[:, -1]

# Sort distances
k_distances_sorted = np.sort(k_distances)

# Plot
plt.plot(k_distances_sorted)
plt.title(f"k-Distance Graph (k={k})")
plt.xlabel("Points sorted by distance")
plt.ylabel(f"Distance to {k}th nearest neighbor")
plt.show()

In [ ]:
# We can run DBSCAN on the chosen epsilon

from sklearn.cluster import DBSCAN

eps_value = 1.25

dbscan = DBSCAN(eps=eps_value, min_samples=k)
labels = dbscan.fit_predict(X)

# Plot results
plt.scatter(X[:, 0], X[:, 1], c=labels)
plt.title(f"DBSCAN Clustering (eps={eps_value})")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
## YOUR TURN 
## Run this method with our real data to test out the right epsilon
